In [ ]:
import requests
import pandas as pd
import io
import os
import json

# ==============================================================================
# CONFIGURATION FOR MULTIPLE DATASETS
# ==============================================================================
TARGET_DATASETS = [
    {
        "file_name": "OECD_RD_GDP_2000_2025.csv",
        "agency_id": "OECD.STI.STP",
        "dataflow_id": "DSD_MSTI@DF_MSTI",
        "sdmx_query_key": "all",  # Small dataset, safe to download "all" and filter via Pandas
        "filters": {
            "UNIT_MEASURE": "PT_B1GQ",
            "MEASURE": "G"
        }
    },
    {
        "file_name": "OECD_Inflation_CPI_2000_2025.csv",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PRICES@DF_PRICES_ALL",
        "sdmx_query_key": ".A.N.CPI.._T.N.GY",
        "filters": {} 
    },
    {
        "file_name": "OECD_Unemployment_Rate_2000_2025.csv",
        "agency_id": "OECD.ELS.SAE",
        "dataflow_id": "DSD_LFS@DF_LFS_INDIC",
        "sdmx_query_key": "all",  
        "filters": {
            "MEASURE": "UNE_RATE",    # Unemployment Rate
            "UNIT_MEASURE": "PT_LF_SUB", # % of Labour Force
            "SEX": "_T",              # Total (Male + Female)
            "AGE": "_T"               # Total Age
        }
    },
    {
        "file_name": "OECD_Productivity_Variation_2000_2025.csv",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PDB@DF_PDB_LV",
        "sdmx_query_key": "all",  
        "filters": {
            "MEASURE": "GDPHRS"  # GDP per hour worked (Labour Productivity)
        }
    },
    {
        "file_name": "OECD_Public_Debt_GDP_2000_2025.csv",
        "agency_id": "OECD.GOV.GIP",
        "dataflow_id": "DSD_GOV@DF_GOV_PF_YU",
        "sdmx_query_key": "A..GGD.PT_B1GQ...",  
        "filters": {}
    }, 
    {
        "file_name": "OECD_Tertiary_Education_2000_2025.csv",
        "agency_id": "OECD.EDU.IMEP",
        "dataflow_id": "DSD_EAG_LSO_EA@DF_LSO_NEAC_DISTR_EA",
        "sdmx_query_key": "._T.Y25T64.ISCED11A_5T8..........OBS...A",  
        "filters": {} # Left empty because the API query key handles all the filtering on the server
    },
    {   "file_name": "OECD_GDP_Growth_2000_2025.csv",
        "agency_id": "OECD.SDD.NAD",
        "dataflow_id": "DSD_NAMAIN1@DF_QNA_EXPENDITURE_GROWTH_OECD",
        "sdmx_query_key": "A.Y..S1.S1.B1GQ._Z._Z._Z.PC.L.GY.T0102",
        "filters": {}  # No Pandas-side filtering needed; server-side key is already precise
    }
]

START_PERIOD = "2000"
END_PERIOD = "2025"
VERSION = "all" # Uses the latest dataset version

headers = {
    "Accept": "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "User-Agent": "DataExtractionScript/4.0 (Python/Requests)"
}

# ==============================================================================
# BATCH EXTRACTION PROCESS
# ==============================================================================

for dataset in TARGET_DATASETS:
    print(f"Processing: {dataset['file_name']}...")
    
    # --------------------------------------------------------------------------
    # FILE EXISTENCE CHECK (SKIP IF ALREADY DOWNLOADED)
    # --------------------------------------------------------------------------
    if os.path.exists(dataset['file_name']):
        print(f"  -> File '{dataset['file_name']}' already exists. Skipping download.\n")
        continue

    # Build the specific URL using the server-side query key to prevent memory overload
    query_key = dataset.get('sdmx_query_key', 'all')
    url = f"https://sdmx.oecd.org/public/rest/data/{dataset['agency_id']},{dataset['dataflow_id']},{VERSION}/{query_key}"
    
    params = {
        "startPeriod": START_PERIOD,
        "endPeriod": END_PERIOD,
        "dimensionAtObservation": "AllDimensions",
        "format": "csvfilewithlabels"
    }
    
    response = requests.get(url, params=params, headers=headers)
    
    if response.status_code == 200:
        csv_stream = io.StringIO(response.text)
        df = pd.read_csv(csv_stream, low_memory=False)
        
        # Protect against completely empty API responses
        if 'OBS_VALUE' not in df.columns:
             print(f"  -> Error: No valid data returned by the API for {dataset['file_name']}.")
             continue
             
        df_clean = df.dropna(subset=['OBS_VALUE']).copy()
        
        # Apply local Pandas filters only if they are defined in the dictionary
        for column_name, filter_value in dataset.get('filters', {}).items():
            if column_name in df_clean.columns:
                df_clean = df_clean[df_clean[column_name] == filter_value]
            else:
                print(f"  -> Warning: Column '{column_name}' not found.")
        
        if df_clean.empty:
            print(f"  -> Error: Dataset is empty after applying filters.")
            
            debug_dict = {}
            for col in df.columns:
                if col not in ['OBS_VALUE', 'TIME_PERIOD', 'REF_AREA']:
                    debug_dict[col] = df[col].dropna().unique().tolist()
            
            with open("DEBUG_OECD_CODES.txt", "w") as f:
                f.write(json.dumps(debug_dict, indent=4))
                
            print("  -> Debug file generated: 'DEBUG_OECD_CODES.txt'. Please check the exact codes.\n")
            continue
        
        target_columns = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
        
        if all(col in df_clean.columns for col in target_columns):
            df_tidy = df_clean[target_columns].rename(columns={
                'REF_AREA': 'Country',
                'TIME_PERIOD': 'Year',
                'OBS_VALUE': 'Value'
            })
            
            df_tidy = df_tidy.reset_index(drop=True)
            df_tidy.to_csv(dataset['file_name'], index=False)
            
            print(f"  -> Success! Saved {len(df_tidy)} rows to {dataset['file_name']}.\n")
        else:
            print(f"  -> Structural error: Missing SDMX columns in {dataset['file_name']}.\n")
    else:
        print(f"  -> HTTP Error {response.status_code} for {dataset['file_name']}.\n")

Processing: OECD_RD_GDP_2000_2025.csv...
  -> File 'OECD_RD_GDP_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Inflation_CPI_2000_2025.csv...
  -> File 'OECD_Inflation_CPI_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Unemployment_Rate_2000_2025.csv...
  -> File 'OECD_Unemployment_Rate_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Productivity_Variation_2000_2025.csv...
  -> File 'OECD_Productivity_Variation_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Public_Debt_GDP_2000_2025.csv...
  -> File 'OECD_Public_Debt_GDP_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Tertiary_Education_2000_2025.csv...
  -> File 'OECD_Tertiary_Education_2000_2025.csv' already exists. Skipping download.

Processing: OECD_GDP_Growth_2000_2025.csv...
  -> Success! Saved 1314 rows to OECD_GDP_Growth_2000_2025.csv.

